valid_x,yを用いて、transactionから候補リストを作成する

In [2]:
import pandas as pd

transactions = pd.read_csv(
    "C:/kaggle_nvidia/kaggle_hm/competitions/hm/data/processed/transactions_with_week.csv",
    dtype={
        "article_id": "int32",
        "week": "int8",
        "price": "float32",
        "sales_channel_id": "int8"
    }
)

valid_x = pd.read_csv(
    "C:/kaggle_nvidia/kaggle_hm/competitions/hm/data/processed/valid_x.csv",
    dtype={
        "article_id": "int32",
        "week": "int8",
        "price": "float32",
        "sales_channel_id": "int8"
    }
)

valid_y = pd.read_csv(
    "C:/kaggle_nvidia/kaggle_hm/competitions/hm/data/processed/valid_y.csv",
    dtype={
        "article_id": "int32",
        "week": "int8",
        "price": "float32",
        "sales_channel_id": "int8"
    }
)

print("読み込み完了")
print(f"transactions: {len(transactions)}行")
print(f"valid_x: {len(valid_x)}行")
print(f"valid_y: {len(valid_y)}行")

読み込み完了
transactions: 31788324行
valid_x: 31575041行
valid_y: 213283行


In [3]:
valid_x.head()

,t_dat,customer_id,article_id,price,sales_channel_id,week
0,2018-09-20,3,663713001,0.050831,2,0
1,2018-09-20,3,541518023,0.030492,2,0
2,2018-09-20,8,505221004,0.015237,2,0
3,2018-09-20,8,685687003,0.016932,2,0
4,2018-09-20,8,685687004,0.016932,2,0


先週の上位30件を候補にする

In [6]:
last_week = valid_x["week"].max()
print(f"valid_xの最終週: {last_week}")

top30 = (
    valid_x[valid_x["week"] == last_week]
    .groupby("article_id")["customer_id"]
    .count()
    .reset_index()
    .rename(columns={"customer_id": "count"})
    .sort_values("count", ascending=False)
    .head(30)
)

print(top30)

valid_xの最終週: 103
       article_id  count
16715   909370001   1310
17692   924243001    817
12136   865799006    750
17368   918522001    666
448     448509014    577
4728    751471001    576
17674   923758001    548
7561    809238001    539
5267    762846027    538
17354   918292001    527
7562    809238005    490
17369   918525001    433
17958   929165002    431
17693   924243002    430
17189   915529003    430
2471    673677002    420
17186   915526001    420
16865   910601003    408
3319    706016001    408
17190   915529005    408
10524   850917001    407
11874   863583001    391
15545   896169005    391
3627    715624001    390
17428   919273002    385
16716   909371001    383
7263    805947001    382
4744    751471043    379
11899   863646001    376
17120   914805002    355


In [7]:
# valid_xに登場する全ユーザーを取得
all_users = valid_x["customer_id"].unique()
print(f"ユーザー数: {len(all_users)}")

# 全ユーザー × top30商品の候補ペアを作成
candidate_top30 = pd.DataFrame({
    "customer_id": all_users.repeat(30),
    "article_id": list(top30["article_id"].values) * len(all_users)
})

print(f"候補ペア数: {len(candidate_top30)}")
print(candidate_top30.head(10))

ユーザー数: 1357359
候補ペア数: 40720770
   customer_id  article_id
0            3   909370001
1            3   924243001
2            3   865799006
3            3   918522001
4            3   448509014
5            3   751471001
6            3   923758001
7            3   809238001
8            3   762846027
9            3   918292001


In [8]:
# valid_yと照合してRecallを計算する
# valid_yは「実際に買われた（ユーザー, 商品）ペア」= 正解

# 候補ペアと正解ペアをマージして一致するものを探す
hit = candidate_top30.merge(
    valid_y[["customer_id", "article_id"]],
    on=["customer_id", "article_id"],
    how="inner"
)

# Recall = 正解のうち候補に含まれた割合
recall = len(hit) / len(valid_y)
print(f"正解数: {len(valid_y)}")
print(f"候補に含まれた正解数: {len(hit)}")
print(f"Recall: {recall:.4f}")

正解数: 213283
候補に含まれた正解数: 9859
Recall: 0.0462
